# EcoSort AI — pipeline definitiva (v2)

Dal dataset grezzo al file `.tflite` pronto per il Raspberry Pi.

**Cosa cambia rispetto alla v1**
1. Il modello si sceglie sul **costo medio per rifiuto**, non sulla `val_loss`.
2. **Calibrazione** delle probabilita (temperature scaling): la regola bayesiana si regge su quelle.
3. Etichette dentro `split.json`: niente piu tre passate sul dataset prima di iniziare.
4. Cache su disco, AdamW + cosine decay, augmentation fotometrica.

Ordine: **Passo 0 → 6**, senza saltare celle.


## Passo 0 — Setup


In [ ]:
import tensorflow as tf, sys
print('TensorFlow:', tf.__version__)
print('Python    :', sys.version.split()[0])
gpu = tf.config.list_physical_devices('GPU')
print('GPU       :', gpu or 'NESSUNA — vai su Runtime > Cambia tipo di runtime > T4')
!pip install -q scikit-learn


In [ ]:
# Drive serve per non perdere il lavoro se la sessione cade a meta benchmark
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/ecosort', exist_ok=True)
print('output in /content/drive/MyDrive/ecosort')


### Carica i 5 moduli Python

`ecosort_decisione.py`, `prepara_dataset.py`, `ecosort_benchmark.py`, `train_finale.py`, `converti_tflite.py`.
Sono nella cartella `training/` del repository.


In [ ]:
from google.colab import files
import os
attesi = ['ecosort_decisione.py','prepara_dataset.py','ecosort_benchmark.py',
          'train_finale.py','converti_tflite.py']
mancanti = [f for f in attesi if not os.path.exists(f)]
if mancanti:
    print('carica:', mancanti)
    files.upload()
print('presenti:', [f for f in attesi if os.path.exists(f)])


### Il dataset

Deve stare in `/content/dataset/` con una sottocartella per classe:
`carta_e_cartone/`, `plastica/`, `vetro_e_metallo/`.


In [ ]:
ZIP_SU_DRIVE = '/content/drive/MyDrive/dataset_definitivo.zip'   # <-- il tuo zip
import os, shutil, zipfile
if not os.path.isdir('/content/dataset'):
    with zipfile.ZipFile(ZIP_SU_DRIVE) as z:
        z.extractall('/content/dataset')
sub = sorted(os.listdir('/content/dataset'))
if len(sub) == 1 and os.path.isdir(f'/content/dataset/{sub[0]}'):
    # zip con una cartella radice: appiattisci
    for d in os.listdir(f'/content/dataset/{sub[0]}'):
        shutil.move(f'/content/dataset/{sub[0]}/{d}', f'/content/dataset/{d}')
    os.rmdir(f'/content/dataset/{sub[0]}')
for c in sorted(os.listdir('/content/dataset')):
    print(f'{c:>20}: {len(os.listdir(f"/content/dataset/{c}"))} immagini')


### Le foto scattate dal Pi — il test set che conta

Le 200+ foto della camera dentro la scatola. Sono l'unico modo di misurare il **domain shift**:
luce fissa, sfondo della scatola, angolo della camera. Il dataset web non contiene niente di simile,
ed e' li che i modelli crollano.


In [ ]:
ZIP_FOTO_PI = '/content/drive/MyDrive/foto_pi.zip'   # <-- stessa struttura a cartelle
import os, zipfile
if not os.path.isdir('/content/foto_pi') and os.path.exists(ZIP_FOTO_PI):
    with zipfile.ZipFile(ZIP_FOTO_PI) as z:
        z.extractall('/content/foto_pi')
if os.path.isdir('/content/foto_pi'):
    for c in sorted(os.listdir('/content/foto_pi')):
        print(f'{c:>20}: {len(os.listdir(f"/content/foto_pi/{c}"))} foto')
else:
    print('nessuna foto del Pi: userai --modo classico')


## Passo 1 — Preparazione del dataset

Deduplica (md5 + hash percettivo), split stratificato, controllo di leakage incrociato
fra le foto del Pi e il dataset web. Produce `split.json`.


In [ ]:
import os
PI = '/content/foto_pi' if os.path.isdir('/content/foto_pi') else None
if PI:
    !python prepara_dataset.py --dataset /content/dataset --foto-pi {PI} --modo pi
else:
    !python prepara_dataset.py --dataset /content/dataset --modo classico


## Passo 2 — Benchmark dei backbone

Quattro backbone, stessa identica pipeline, confrontati sul **costo** e non sull'accuratezza.
Circa 2-3 ore su T4. Lo stato e' su Drive: se la sessione cade, rilancia questa cella e
riprende dal backbone successivo.


In [ ]:
!python ecosort_benchmark.py


## Passo 3 — Accuratezza contro costo contro latenza

Il vincitore non e' il modello piu accurato: e' quello che minimizza il costo per rifiuto
restando dentro la finestra di 2-3 secondi del Pi.


In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
BG, ACC, ACC2, MUTED, TXT = '#0A0F0D', '#D4FF3A', '#00D97E', '#9CA3AF', '#E8EFE9'
ris = json.load(open('/content/drive/MyDrive/ecosort/stato_benchmark.json'))['risultati']
ris = sorted(ris, key=lambda r: r['costo_decisione'])
fig, ax = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG)
for a in ax: a.set_facecolor(BG); a.tick_params(colors=TXT); [s.set_color(MUTED) for s in a.spines.values()]
nomi = [r['backbone'] for r in ris]
y = np.arange(len(nomi))
ax[0].barh(y, [r['costo_decisione'] for r in ris], color=ACC)
ax[0].barh(y, [r['costo_argmax'] for r in ris], color=MUTED, alpha=.35)
ax[0].set_yticks(y); ax[0].set_yticklabels(nomi, color=TXT)
ax[0].invert_yaxis(); ax[0].set_xlabel('costo medio per rifiuto', color=TXT)
ax[0].set_title('regola a costo (verde) vs argmax (grigio)', color=TXT)
for r in ris:
    ax[1].scatter(r['latenza_cpu_ms'], r['accuracy']*100, s=90, color=ACC2)
    ax[1].annotate(r['backbone'], (r['latenza_cpu_ms'], r['accuracy']*100),
                   color=TXT, fontsize=8, xytext=(6, 4), textcoords='offset points')
ax[1].set_xlabel('latenza CPU x86 (ms) — proxy del Pi', color=TXT)
ax[1].set_ylabel('accuratezza (%)', color=TXT)
ax[1].set_title('accuratezza contro latenza', color=TXT)
plt.tight_layout(); plt.show()
for r in ris:
    print(f"{r['backbone']:<20} acc {r['accuracy']*100:5.2f}%  costo {r['costo_decisione']:.4f}"
          f"  T {r['temperatura']:.2f}  gravi {r['gravi_decisione']:>3}  {r['latenza_cpu_ms']:.0f} ms")


## Passo 4 — Training finale

La validation ha finito il suo lavoro (scegliere backbone ed epoche). Il modello definitivo
si riaddestra su train + validation, tenendo fuori solo una piccola fetta per ricalibrare
la temperatura. Il test set resta intoccato.


In [ ]:
!python train_finale.py


## Passo 5 — Conversione in TFLite

Tre varianti (float32, float16, INT8), **tutte valutate sul test set** con le probabilita
calibrate. Vince la piu piccola che non perde accuratezza e non aggiunge errori gravi.


In [ ]:
!python converti_tflite.py


## Passo 6 — Pacchetto per il Raspberry Pi


In [ ]:
import shutil, os, json
OUT = '/content/drive/MyDrive/ecosort'
PKG = '/content/pacchetto_pi'
os.makedirs(PKG, exist_ok=True)
for f in ['rifiuti.tflite', 'config.json']:
    shutil.copy(os.path.join(OUT, f), PKG)
shutil.copy('ecosort_decisione.py', PKG)
if os.path.exists('classifica_pi.py'):
    shutil.copy('classifica_pi.py', PKG)
shutil.make_archive('/content/pacchetto_pi', 'zip', PKG)
print(json.dumps(json.load(open(os.path.join(OUT, 'config.json'))), indent=2))
from google.colab import files
files.download('/content/pacchetto_pi.zip')
files.download(os.path.join(OUT, 'newbest_model.keras'))   # per l'app desktop


## Sul Raspberry Pi

```bash
pip install ai-edge-litert numpy pillow --break-system-packages
unzip pacchetto_pi.zip -d ~/ecosort && cd ~/ecosort
python3 classifica_pi.py --bench            # latenza reale
python3 classifica_pi.py --foto prova.jpg
python3 classifica_pi.py                    # INVIO per scattare
```

`newbest_model.keras` va invece in `desktop-app/server/` dell'app JavaFX.
